# NAMD-FEP — notebook 4 · run the production job and watch it

This notebook is the control room. Two halves:

1. **Launch** — how the production run is started on a GPU machine (`rbfe run`,
   normally under the self-healing `respawn_controller.sh`). **A free Binder session
   has no GPU**, so the launch cell is a guarded no-op there that prints the command
   to run elsewhere.
2. **Monitor** — works everywhere, including Binder, on any system directory that
   has run output: `rbfe status`, per-window progress bars, and a live dE trace of
   the windows that are done. The shipped **6I5I example** has a complete run on
   disk, so the whole monitoring half can be exercised immediately.

The job being watched: 16 λ values → **15 windows** per direction; each leg
(`complex`, `solvent`) runs **forward and backward** — 60 window-runs in total.
The runner is marker-guarded: each finished window drops a `.done` marker, and any
interruption costs at most the window in flight.

## 4.1 Setup — pick the system to watch

The tutorial system from notebooks 02–03 if it exists, else the shipped 6I5I
example. Pointing this notebook at `6I5I_DUAL_FEP` needs one caveat: that tree
predates `system.ini`, so a tiny shim config is written into `_tutorial/` mapping
the legacy directory at it — **the legacy tree is never written to.**

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

REPO = Path.cwd().resolve()
if not (REPO / "bin" / "rbfe").exists():
    REPO = REPO.parent
os.environ["PATH"] = str(REPO / "bin") + os.pathsep + os.environ["PATH"]
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

TUTORIAL = REPO / "_tutorial" / "6i5i"
LEGACY   = REPO / "6I5I_DUAL_FEP"

if (TUTORIAL / "system.ini").exists():
    SYS = TUTORIAL
else:
    # A read-only view of the legacy example: an ini in _tutorial that points
    # at 6I5I_DUAL_FEP's inputs. Run output already lives in the legacy tree,
    # and rbfe reads it from there; nothing is written back.
    SYS = REPO / "_tutorial" / "legacy_6i5i"
    SYS.mkdir(parents=True, exist_ok=True)
    (SYS / "system.ini").write_text(f"""\
[system]
name        = 6i5i (legacy example)
description = read-only monitor view of 6I5I_DUAL_FEP
[ligand]
dir     = {LEGACY}/inputs
ref     = ref
mut     = mut
resname = UNL
[protein]
pdb     = {LEGACY}/inputs/protein.pdb
chain   = A
[mutation]
strategy = atom_addition
vanish   = C12 H7 H8 H9
appear   = H
[run]
legs             = complex solvent
lambdas          = 0.0 0.045 0.09 0.14546 0.22425 0.30303 0.38182 0.46061
                    0.5394 0.61819 0.697 0.77576 0.85455 0.91 0.955 1.0
steps_per_window = 250000
alch_equil_steps = 50000
outfreq          = 500
temperature      = 300.0
""")

print("watching:", SYS)
assert (SYS / "system.ini").exists()

## 4.2 Launch — GPU machines only

The way to run: **one NAMD invocation per λ window**, wrapped in a controller that
relaunches it after a node kill. From a shell on the GPU box:

```bash
cd <system dir>
rbfe run --respawn        # the self-healing wrapper (recommended)
rbfe run                  # or in the foreground: every leg and direction, in order
rbfe run complex forward  # or exactly one direction
```

The leg order is fixed — complex forward → complex backward → solvent forward →
solvent backward — because a crash before the cheap leg costs least.

The cell below launches **only** when NAMD is really there and the configs exist;
on Binder it prints the command and exits.

In [ ]:
NAMD = os.environ.get("NAMD") or shutil.which("namd3")
ready = NAMD and (SYS / "fep.tcl").exists()

if not ready:
    print("NAMD or the generated configs are not here -- nothing to launch.")
    print(f"On the GPU box:  cd {SYS} && rbfe run --respawn")
else:
    r = subprocess.run(["rbfe", "run", "--respawn"], cwd=SYS,
                       capture_output=True, text=True)
    print(r.stdout, r.stderr)

## 4.3 `rbfe status` — where the run is

Completion is judged by the per-window `.done` markers, never by the existence of a
`.fepout` (a killed window leaves a partial file behind). Re-run this cell as often
as you like — it is read-only.

In [ ]:
r = subprocess.run(["rbfe", "status"], cwd=SYS, capture_output=True, text=True)
print(r.stdout or r.stderr)

## 4.4 Progress as a figure

The same data, drawn: one row per leg/direction, one cell per window. Dark cells are
done; the outlined cell is the one in flight.

In [ ]:
import json
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

r = subprocess.run(["rbfe", "status", "--json"], cwd=SYS,
                   capture_output=True, text=True)
try:
    st = json.loads(r.stdout)
except ValueError:
    st = None
    print("no machine-readable status yet (nothing on disk) --", r.stderr.strip())

if st:
    nwin, rows = st["nwin"], st["windows"]
    fig, ax = plt.subplots(figsize=(10, 1.1 * len(rows)))
    grid = np.zeros((len(rows), nwin))
    labels = []
    for i, row in enumerate(rows):
        done = row["per_window_done"]
        grid[i, :done] = 1.0
        labels.append(f"{row['leg']} {row['direction']}  {done}/{nwin}")
    ax.imshow(grid, aspect="auto", cmap="Greens", vmin=0, vmax=1)
    ax.set_yticks(range(len(rows)), labels=labels)
    ax.set_xticks(range(nwin), labels=[f"w{i:02d}" for i in range(nwin)],
                  rotation=90)
    ax.set_title(st["system"])
    ax.set_xlabel("lambda window")
    plt.tight_layout()
    plt.show()

## 4.5 Preliminary numbers while it runs

Each finished window is already a free-energy sample. This plots the one-sided EXP
estimate per finished window — biased until the backward direction exists (the final
number comes from BAR in notebook 05), but it answers the live question: *is the dE
signal sane, window by window?*

The extraction is the same one the audit uses (`rbfe.analysis.parse_windows`),
trimmed to post-equilibration samples.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from rbfe import analysis as A
from rbfe.config import load as load_config

cfg = load_config(SYS / "system.ini")
A.configure(cfg)

fig, axes = plt.subplots(1, len(cfg.legs), figsize=(6 * len(cfg.legs), 3.6),
                         sharey=True, squeeze=False)
plotted = False
for ax, leg in zip(axes[0], cfg.legs):
    for direction, colour in (("forward", "tab:blue"), ("backward", "tab:red")):
        wins = []
        for i in range(cfg.nwin):
            f = SYS / leg / f"md_{direction}_w{i:02d}.fepout"
            if f.exists():
                w = A.parse_windows(f)
                if w:
                    w = w[0]
                    dE = w["all"][w["prod_start"]:]
                    wins.append((i, A.exp_1sided(dE)))
        if wins:
            plotted = True
            xs, ys = zip(*wins)
            ax.plot(xs, ys, "o-", color=colour, label=direction,
                    markersize=4)
    ax.axhline(0, color="grey", lw=0.8)
    ax.set_title(f"{leg} leg")
    ax.set_xlabel("window")
    ax.legend(frameon=False)
axes[0][0].set_ylabel("one-sided EXP ΔG per window (kcal/mol)")
plt.tight_layout()
if plotted:
    plt.show()
    print("preview only -- the final ΔΔG is BAR over both directions (notebook 05)")
else:
    print("no per-window .fepout files found yet")

## 4.6 The watchdog from a notebook

For a run left on a GPU box, `rbfe run --watchdog 300` snapshots the status every
five minutes into `checkpoint_status.log`. From here, re-running 4.3–4.5 is the
interactive equivalent.

**When every direction is complete** (`all directions complete -- run rbfe audit`
in 4.3): assemble the combined files with `rbfe assemble complex md_forward` etc.
if you want them, or go straight to **`05_equilibration_and_analysis.ipynb`**, which
reads the per-window files directly — as the audit does.